# Train SARO's PAS policy for Go2 on Kaggle (2x T4)

Replicates the low-level locomotion training technique from **SARO** (arXiv:2407.16412) for the Unitree Go2, using [mjlab](https://mujocolab.github.io/mjlab/) + `rsl_rl`. Implementation + architecture docs: `docs/07-pas-implementation.md` in the repo.

**Before running:**
1. Notebook settings (right sidebar) -> **Accelerator: GPU T4 x2**, **Internet: On**.
2. Add-ons -> Secrets -> add a secret named `HF_TOKEN` with a Hugging Face **write** token (huggingface.co/settings/tokens). Used to create/push checkpoints to a model repo, and to resume across sessions (Kaggle sessions don't persist disk).
3. If `github.com/BRUH-MAIN/policyswitching` is private, also add a secret `GITHUB_TOKEN` (a GitHub PAT with `repo` scope).
4. Edit the CONFIG cell below if you want a different HF repo name, env count, or iteration budget.

**What this notebook does:** clones the repo, installs mjlab/rsl_rl, checks Hugging Face for an existing checkpoint to resume from (so re-running this notebook after a session gets killed picks up where it left off), trains Stage 1 (oracle) then Stage 2 (anneal), uploading every checkpoint to your HF model repo as it trains.

In [ ]:
# ==== CONFIG — edit as needed ====
GITHUB_REPO = "BRUH-MAIN/policyswitching"
HF_REPO_NAME = "go2-pas-saro"       # final repo id will be f"{your_hf_username}/{HF_REPO_NAME}"

NUM_ENVS = 4096                       # 2x T4 (16GB each) — reduce if you hit OOM, see the env-check cell's guidance
GPU_IDS = "0 1"                       # "0" for a single GPU, "0 1" for both T4s (torchrunx multi-GPU)
STAGE1_MAX_ITERATIONS = 40000          # paper default; lower this for a first smoke run, e.g. 2000
STAGE2_MAX_ITERATIONS = 40000
SAVE_INTERVAL = 200                    # PPO iterations between checkpoints (and HF uploads)

REPO_DIR = "/kaggle/working/policyswitching"
MJLAB_DIR = f"{REPO_DIR}/unitree_rl_mjlab"

## 1. Environment check
Fail fast here rather than after a slow install — mjlab needs driver ≥550 and CUDA 12.4+ (MuJoCo Warp is picky about CUDA version).

In [ ]:
!nvidia-smi
!nvcc --version || echo 'nvcc not found (fine if a matching CUDA runtime is still installed via pip)'
!python3 --version

## 2. Clone the repo

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        github_token = None

    if github_token:
        clone_url = f"https://{github_token}@github.com/{GITHUB_REPO}.git"
    else:
        clone_url = f"https://github.com/{GITHUB_REPO}.git"
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

!ls {MJLAB_DIR}/src/tasks/velocity/mdp/pas.py && echo 'PAS implementation found.'

## 3. Install dependencies
Kaggle's Python env is used directly (no conda needed — unlike local dev, there's no conflicting editable install here to worry about).

In [ ]:
%cd {MJLAB_DIR}
!pip install -q -e .
# setup.py pins mjlab==1.2.0 / mujoco-warp==3.5.0 but not an exact mujoco (core) version.
# Kaggle images ship an older mujoco that already satisfies mujoco-warp's loose bound, so
# pip never upgrades it -- force the matching version explicitly (verified locally that
# mujoco==3.5.0 is what mujoco-warp==3.5.0 actually needs; bump both together if you ever
# change MJLAB/mujoco-warp versions).
!pip install -q --upgrade "mujoco==3.5.0"
!pip install -q huggingface_hub

import mjlab, mujoco, mujoco_warp
print("mujoco:", mujoco.__version__)
print("mujoco_warp OK:", mujoco_warp.__file__)
print("mjlab OK:", mjlab.__file__)

## 4. Hugging Face login + cross-session checkpoint sync
Kaggle sessions don't persist disk between runs, so every checkpoint is uploaded to a Hugging Face model repo as training goes (via `HF_CHECKPOINT_REPO` env var read by `PasOnPolicyRunner.save()`), and this notebook checks that repo for a checkpoint to resume from before starting.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, whoami, create_repo, hf_hub_download

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

hf_username = whoami()["name"]
HF_REPO_ID = f"{hf_username}/{HF_REPO_NAME}"
create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)
print("Using Hugging Face model repo:", HF_REPO_ID)

os.environ["HF_CHECKPOINT_REPO"] = HF_REPO_ID


def latest_hf_checkpoint(stage: str):
    """Return (iteration, filename) for the highest-iteration model_*.pt under {stage}/ on HF, or None."""
    api = HfApi()
    files = api.list_repo_files(HF_REPO_ID, repo_type="model")
    candidates = []
    for f in files:
        if f.startswith(f"{stage}/model_") and f.endswith(".pt"):
            try:
                it = int(f.split("model_")[-1].split(".pt")[0])
                candidates.append((it, f))
            except ValueError:
                continue
    if not candidates:
        return None
    return max(candidates, key=lambda x: x[0])


def sync_from_hf(stage: str, local_run_name: str):
    """Download the latest {stage} checkpoint from HF into a fixed local run dir.

    Returns the `--agent.*` resume args to pass to train.py, or [] if nothing
    was found on HF (i.e. this stage hasn't started yet).
    """
    found = latest_hf_checkpoint(stage)
    if found is None:
        print(f"No existing '{stage}' checkpoint on HF -- starting fresh.")
        return []
    iteration, remote_path = found
    local_dir = f"{MJLAB_DIR}/logs/rsl_rl/go2_pas/{local_run_name}"
    os.makedirs(local_dir, exist_ok=True)
    local_file = hf_hub_download(HF_REPO_ID, remote_path, repo_type="model", local_dir=MJLAB_DIR)
    target = f"{local_dir}/model_{iteration}.pt"
    if local_file != target:
        import shutil
        shutil.copy(local_file, target)
    print(f"Resuming '{stage}' from iteration {iteration} ({remote_path}).")
    return [
        "--agent.resume", "True",
        "--agent.load-run", local_run_name,
        "--agent.load-checkpoint", f"model_{iteration}.pt",
    ]

## 5. Stage 1 — Oracle
`anneal_prob` pinned at 1.0. Trains the terrain encoder + actor MLP via PPO, and the estimator via an auxiliary reconstruction loss only (see `docs/07-pas-implementation.md` §2).

In [ ]:
os.environ["HF_CHECKPOINT_STAGE"] = "stage1"
os.environ["MUJOCO_GL"] = "egl"

stage1_resume_args = sync_from_hf("stage1", "kaggle_stage1")

%cd {MJLAB_DIR}
!python scripts/train.py Unitree-Go2-PAS-Oracle \
  --env.scene.num-envs {NUM_ENVS} \
  --agent.max-iterations {STAGE1_MAX_ITERATIONS} \
  --agent.save-interval {SAVE_INTERVAL} \
  --agent.logger tensorboard \
  --agent.experiment-name go2_pas \
  --gpu-ids {GPU_IDS} \
  {' '.join(stage1_resume_args)}

## 6. Stage 2 — Anneal
Resumes from Stage 1's final checkpoint (or continues an interrupted Stage 2 run if one already exists on HF). `anneal_prob` decays as `0.9998^iteration`, restarting fresh each time a new Stage-2 run starts (see §7 of the doc — this matters if Stage 2 itself gets interrupted and resumed).

In [ ]:
os.environ["HF_CHECKPOINT_STAGE"] = "stage2"

stage2_resume_args = sync_from_hf("stage2", "kaggle_stage2")
if not stage2_resume_args:
    # First time entering Stage 2: resume from Stage 1's final checkpoint instead.
    stage2_resume_args = sync_from_hf("stage1", "kaggle_stage1")

%cd {MJLAB_DIR}
!python scripts/train.py Unitree-Go2-PAS-Anneal \
  --env.scene.num-envs {NUM_ENVS} \
  --agent.max-iterations {STAGE2_MAX_ITERATIONS} \
  --agent.save-interval {SAVE_INTERVAL} \
  --agent.logger tensorboard \
  --agent.experiment-name go2_pas \
  --agent.run-name stage2 \
  --gpu-ids {GPU_IDS} \
  {' '.join(stage2_resume_args)}

## 7. Next steps

- **Monitor**: `%load_ext tensorboard` then `%tensorboard --logdir {MJLAB_DIR}/logs/rsl_rl/go2_pas` in a cell, or watch the `Mean reward` / `anneal_prob` / `estimator_mse` lines printed above.
- **If the session gets killed**: just re-run this notebook top to bottom. Both stage cells check HF first and resume automatically — nothing is lost except whatever happened since the last `SAVE_INTERVAL` checkpoint.
- **Checkpoints** live at `hf.co/{HF_REPO_ID}` under `stage1/` and `stage2/`.
- **Visual evaluation** (needs a display / won't run headless on Kaggle) — download a checkpoint and run locally:
  ```bash
  python scripts/play.py Unitree-Go2-PAS-Anneal --checkpoint_file <downloaded model_N.pt>
  ```
- **ONNX/deployment export** is not yet PAS-aware (see `docs/07-pas-implementation.md` §7) — the `.pt` checkpoints here are for resuming/evaluating, not yet for flashing onto a real robot.